# MLflow → Triton: Colab quick start

Train a small scikit-learn classifier and save it to MLflow together with Triton serving files. Run the cells in order using a CPU runtime; no GPU or Triton server is required.

This notebook uses scikit-learn's bundled Iris dataset. Generated serving code is intended for demos and proofs of concept; validate and optimize it before production use.


In [ ]:
%pip install -q mlflow-triton scikit-learn


## Train and log a model

This example uses a local MLflow store inside the Colab runtime. Its files disappear when the runtime is deleted. For an existing remote MLflow server, replace the tracking configuration with your own before logging.


In [ ]:
from pathlib import Path

import mlflow
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from mlflow_triton import sklearn

workspace = Path("/content/mlflow-triton-demo")
workspace.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{workspace / 'mlflow.db'}")
experiment_name = "triton-colab-demo"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(
        experiment_name,
        artifact_location=(workspace / "artifacts").as_uri(),
    )
mlflow.set_experiment(experiment_name)

inputs, targets = load_iris(return_X_y=True, as_frame=True)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200))
model.fit(inputs, targets)

info = sklearn.log_model(
    model,
    name="iris_classifier",
    input_example=inputs.iloc[:2],
)
print(f"Model and Triton serving files saved to: {info.model_uri}")


Logging is complete. The trained model remains in memory, and MLflow now stores the model and generated Triton files. Logging does not deploy a service or start Triton.

## Download serving files (optional)

Run this cell only if you want to take the Triton files out of Colab. `triton.load_model()` retrieves the files as a local directory; it does not reload model weights. The ZIP contains `model_repository/iris_classifier/`, ready to hand off to your deployment tooling.


In [ ]:
import shutil
from google.colab import files

from mlflow_triton import triton

model_directory = triton.load_model(info.model_uri)
export_root = workspace / "export"
repository = export_root / "model_repository"
repository.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    model_directory,
    repository / model_directory.name,
    dirs_exist_ok=True,
)
archive = shutil.make_archive(
    str(workspace / "triton-serving-files"),
    "zip",
    root_dir=export_root,
    base_dir="model_repository",
)
files.download(archive)


The ZIP contains serving files, not the local MLflow tracking database. Install the recorded dependencies in an environment compatible with Triton's Python backend before serving.

- [Python API guide](https://github.com/ai-lab-tech/mlflow-plugin/blob/main/packages/mlflow-triton/README.md)
- [Optional Triton Control deployment](https://github.com/ai-lab-tech/mlflow-plugin/blob/main/packages/mlflow-triton-control/README.md)
- [Send an example request after deployment](https://github.com/ai-lab-tech/mlflow-plugin#send-an-example-request)
